In [1]:
from pathlib import Path

import pandas as pd
from elasticsearch import Elasticsearch, helpers
from sentence_transformers import SentenceTransformer

In [2]:

DATA_PATH = Path("../data/processed/bev_vehicles.csv")

ELASTICSEARCH_URL = "http://localhost:9200"
INDEX_NAME = "ev-vehicles"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

EMBEDDING_DIMENSIONS = 384

In [3]:
es_client = Elasticsearch(ELASTICSEARCH_URL)

print(es_client.info())

{'name': '3016a6d3053d', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'Ex3VVj5uSn-tVMlx6zVcpg', 'version': {'number': '8.15.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '1a77947f34deddb41af25e6f0ddb8e830159c179', 'build_date': '2024-08-05T10:05:34.233336849Z', 'build_snapshot': False, 'lucene_version': '9.11.1', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [4]:
df = pd.read_csv(DATA_PATH)

print(f"Number of BEV records: {len(df):,}")
print(f"Number of columns: {len(df.columns)}")

df.head()

Number of BEV records: 1,572
Number of columns: 20


,id,year,make,model,vehicle_class,drive,transmission,electric_range_miles,city_mpge,highway_mpge,combined_mpge,charge_120v_hours,charge_240v_hours,ev_motor,annual_energy_cost_usd,tailpipe_co2_gpm,vehicle_type,fuel_type,vehicle_name,document_text
0,16423,2000,Nissan,Altra EV,Midsize Station Wagons,NaN,NaN,90.0,81.0,91.0,85.0,NaN,NaN,62 KW AC Induction,900.0,0.0,Battery Electric Vehicle (BEV),Electricity,2000 Nissan Altra EV,Vehicle: 2000 Nissan Altra EV\nFuelEconomy.gov...
1,16424,2000,Toyota,RAV4 EV,Sport Utility Vehicle - 2WD,2-Wheel Drive,NaN,88.0,81.0,64.0,72.0,NaN,NaN,50 KW DC,1050.0,0.0,Battery Electric Vehicle (BEV),Electricity,2000 Toyota RAV4 EV,Vehicle: 2000 Toyota RAV4 EV\nFuelEconomy.gov ...
2,17328,2001,Toyota,RAV4 EV,Sport Utility Vehicle - 2WD,2-Wheel Drive,NaN,88.0,81.0,64.0,72.0,NaN,NaN,50 KW DC,1050.0,0.0,Battery Electric Vehicle (BEV),Electricity,2001 Toyota RAV4 EV,Vehicle: 2001 Toyota RAV4 EV\nFuelEconomy.gov ...
3,17329,2001,Ford,Th!nk,Two Seaters,NaN,NaN,29.0,74.0,58.0,65.0,NaN,NaN,27 KW AC Induction,1150.0,0.0,Battery Electric Vehicle (BEV),Electricity,2001 Ford Th!nk,Vehicle: 2001 Ford Th!nk\nFuelEconomy.gov vehi...
4,17330,2001,Ford,Explorer USPS Electric,Sport Utility Vehicle - 2WD,2-Wheel Drive,NaN,38.0,45.0,33.0,39.0,NaN,NaN,67 KW AC Induction,1950.0,0.0,Battery Electric Vehicle (BEV),Electricity,2001 Ford Explorer USPS Electric,Vehicle: 2001 Ford Explorer USPS Electric\nFue...


In [5]:
df.columns.tolist()

['id',
 'year',
 'make',
 'model',
 'vehicle_class',
 'drive',
 'transmission',
 'electric_range_miles',
 'city_mpge',
 'highway_mpge',
 'combined_mpge',
 'charge_120v_hours',
 'charge_240v_hours',
 'ev_motor',
 'annual_energy_cost_usd',
 'tailpipe_co2_gpm',
 'vehicle_type',
 'fuel_type',
 'vehicle_name',
 'document_text']

In [6]:
print(df.loc[0, "document_text"])

Vehicle: 2000 Nissan Altra EV
FuelEconomy.gov vehicle ID: 16423
Vehicle type: Battery Electric Vehicle (BEV)
Vehicle class: Midsize Station Wagons
Drive: 
Transmission: 
Electric range: 90.0 miles
City efficiency: 81.0 MPGe
Highway efficiency: 91.0 MPGe
Combined efficiency: 85.0 MPGe
120V charging time: nan hours
240V charging time: nan hours
Electric motor: 62 KW AC Induction
Annual energy cost: 900.0 USD
Tailpipe CO2 emissions: 0.0 grams per mile
Source: U.S. Department of Energy FuelEconomy.gov dataset.


In [7]:
# Defining the Elasticsearch index mapping

index_settings = {
    "number_of_shards": 1,
    "number_of_replicas": 0,
}

index_mapping = {
    "properties": {
        "id": {"type": "keyword"},
        "year": {"type": "integer"},
        "make": {
            "type": "text",
            "fields": {
                "keyword": {
                    "type": "keyword",
                    "ignore_above": 256,
                }
            },
        },
        "model": {
            "type": "text",
            "fields": {
                "keyword": {
                    "type": "keyword",
                    "ignore_above": 256,
                }
            },
        },
        "vehicle_name": {
            "type": "text",
            "fields": {
                "keyword": {
                    "type": "keyword",
                    "ignore_above": 512,
                }
            },
        },
        "vehicle_type": {"type": "keyword"},
        "fuel_type": {"type": "keyword"},
        "vehicle_class": {"type": "text"},
        "drive": {"type": "text"},
        "transmission": {"type": "text"},
        "electric_range_miles": {"type": "float"},
        "city_mpge": {"type": "float"},
        "highway_mpge": {"type": "float"},
        "combined_mpge": {"type": "float"},
        "charge_120v_hours": {"type": "float"},
        "charge_240v_hours": {"type": "float"},
        "ev_motor": {"type": "text"},
        "annual_fuel_cost_usd": {"type": "float"},
        "tailpipe_co2_gpm": {"type": "float"},
        "document_text": {"type": "text"},
        "embedding": {
            "type": "dense_vector",
            "dims": EMBEDDING_DIMENSIONS,
            "index": True,
            "similarity": "cosine",
        },
    }
}

In [8]:
"""
Create or recreate the Elasticsearch index
This cell deletes the existing index if it exists.

That is useful during development because each rerun starts from a clean index.

"""


if es_client.indices.exists(index=INDEX_NAME):
    es_client.indices.delete(index=INDEX_NAME)
    print(f"Deleted existing index: {INDEX_NAME}")

es_client.indices.create(
    index=INDEX_NAME,
    settings=index_settings,
    mappings=index_mapping,
)

print(f"Created index: {INDEX_NAME}")

Created index: ev-vehicles


In [9]:
es_client.indices.get_mapping(index=INDEX_NAME)

ObjectApiResponse({'ev-vehicles': {'mappings': {'properties': {'annual_fuel_cost_usd': {'type': 'float'}, 'charge_120v_hours': {'type': 'float'}, 'charge_240v_hours': {'type': 'float'}, 'city_mpge': {'type': 'float'}, 'combined_mpge': {'type': 'float'}, 'document_text': {'type': 'text'}, 'drive': {'type': 'text'}, 'electric_range_miles': {'type': 'float'}, 'embedding': {'type': 'dense_vector', 'dims': 384, 'index': True, 'similarity': 'cosine', 'index_options': {'type': 'int8_hnsw', 'm': 16, 'ef_construction': 100}}, 'ev_motor': {'type': 'text'}, 'fuel_type': {'type': 'keyword'}, 'highway_mpge': {'type': 'float'}, 'id': {'type': 'keyword'}, 'make': {'type': 'text', 'fields': {'keyword': {'type': 'keyword', 'ignore_above': 256}}}, 'model': {'type': 'text', 'fields': {'keyword': {'type': 'keyword', 'ignore_above': 256}}}, 'tailpipe_co2_gpm': {'type': 'float'}, 'transmission': {'type': 'text'}, 'vehicle_class': {'type': 'text'}, 'vehicle_name': {'type': 'text', 'fields': {'keyword': {'typ

In [10]:
# Load the Sentence Transformer model

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print(f"Loaded model: {EMBEDDING_MODEL_NAME}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded model: sentence-transformers/all-MiniLM-L6-v2


In [11]:
test_embedding = embedding_model.encode(
    "What BEVs have the longest driving range?",
    normalize_embeddings=True,
)

print(f"Embedding dimensions: {len(test_embedding)}")

Embedding dimensions: 384


In [12]:
# Generate embeddings for BEV documents

documents = df["document_text"].tolist()

embeddings = embedding_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(f"Number of documents: {len(documents)}")
print(f"Number of embeddings: {len(embeddings)}")
print(f"Dimensions per embedding: {len(embeddings[0])}")

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

Number of documents: 1572
Number of embeddings: 1572
Dimensions per embedding: 384


In [13]:
assert len(documents) == len(embeddings)
assert len(embeddings[0]) == EMBEDDING_DIMENSIONS

print("Embedding validation passed.")

Embedding validation passed.


In [14]:
"""
Prepare documents for Elasticsearch

Elasticsearch does not accept pandas NaN values cleanly, 
so we convert them to Python None.
"""

def clean_value(value):
    if pd.isna(value):
        return None

    if hasattr(value, "item"):
        return value.item()

    return value

In [15]:
# Creating documents for bulk indexing

actions = []

for row, embedding in zip(df.to_dict(orient="records"), embeddings):
    document = {
        field_name: clean_value(value)
        for field_name, value in row.items()
    }

    document["id"] = str(document["id"])
    document["embedding"] = embedding.tolist()

    action = {
        "_index": INDEX_NAME,
        "_id": document["id"],
        "_source": document,
    }

    actions.append(action)

print(f"Prepared {len(actions):,} Elasticsearch documents.")

Prepared 1,572 Elasticsearch documents.


In [16]:
actions[0]["_source"]

{'id': '16423',
 'year': 2000,
 'make': 'Nissan',
 'model': 'Altra EV',
 'vehicle_class': 'Midsize Station Wagons',
 'drive': None,
 'transmission': None,
 'electric_range_miles': 90.0,
 'city_mpge': 81.0,
 'highway_mpge': 91.0,
 'combined_mpge': 85.0,
 'charge_120v_hours': None,
 'charge_240v_hours': None,
 'ev_motor': '62 KW AC Induction',
 'annual_energy_cost_usd': 900.0,
 'tailpipe_co2_gpm': 0.0,
 'vehicle_type': 'Battery Electric Vehicle (BEV)',
 'fuel_type': 'Electricity',
 'vehicle_name': '2000 Nissan Altra EV',
 'document_text': 'Vehicle: 2000 Nissan Altra EV\nFuelEconomy.gov vehicle ID: 16423\nVehicle type: Battery Electric Vehicle (BEV)\nVehicle class: Midsize Station Wagons\nDrive: \nTransmission: \nElectric range: 90.0 miles\nCity efficiency: 81.0 MPGe\nHighway efficiency: 91.0 MPGe\nCombined efficiency: 85.0 MPGe\n120V charging time: nan hours\n240V charging time: nan hours\nElectric motor: 62 KW AC Induction\nAnnual energy cost: 900.0 USD\nTailpipe CO2 emissions: 0.0 gram

In [17]:
# Bulk index all BEV documents

success_count, _ = helpers.bulk(
    es_client,
    actions,
)

es_client.indices.refresh(index=INDEX_NAME)

print(f"Successfully indexed {success_count} BEV documents.")

Successfully indexed 1572 BEV documents.


In [18]:
count_result = es_client.count(index=INDEX_NAME)

print(f"Elasticsearch document count: {count_result['count']:,}")
print(f"Processed CSV record count: {len(df):,}")

Elasticsearch document count: 1,572
Processed CSV record count: 1,572


In [19]:
# Inspect one indexed document

result = es_client.search(
    index=INDEX_NAME,
    size=1,
    source_excludes=["embedding"],
    query={
        "match_all": {}
    },
)

result["hits"]["hits"][0]

{'_index': 'ev-vehicles',
 '_id': '16423',
 '_score': 1.0,
 '_source': {'id': '16423',
  'year': 2000,
  'make': 'Nissan',
  'model': 'Altra EV',
  'vehicle_class': 'Midsize Station Wagons',
  'drive': None,
  'transmission': None,
  'electric_range_miles': 90.0,
  'city_mpge': 81.0,
  'highway_mpge': 91.0,
  'combined_mpge': 85.0,
  'charge_120v_hours': None,
  'charge_240v_hours': None,
  'ev_motor': '62 KW AC Induction',
  'annual_energy_cost_usd': 900.0,
  'tailpipe_co2_gpm': 0.0,
  'vehicle_type': 'Battery Electric Vehicle (BEV)',
  'fuel_type': 'Electricity',
  'vehicle_name': '2000 Nissan Altra EV',
  'document_text': 'Vehicle: 2000 Nissan Altra EV\nFuelEconomy.gov vehicle ID: 16423\nVehicle type: Battery Electric Vehicle (BEV)\nVehicle class: Midsize Station Wagons\nDrive: \nTransmission: \nElectric range: 90.0 miles\nCity efficiency: 81.0 MPGe\nHighway efficiency: 91.0 MPGe\nCombined efficiency: 85.0 MPGe\n120V charging time: nan hours\n240V charging time: nan hours\nElectric 

In [20]:
query = "Tesla Model 3"

search_result = es_client.search(
    index=INDEX_NAME,
    size=5,
    source=[
        "vehicle_name",
        "electric_range_miles",
        "combined_mpge",
        "vehicle_class",
    ],
    query={
        "multi_match": {
            "query": query,
            "fields": [
                "vehicle_name^3",
                "make^2",
                "model^2",
                "document_text",
            ],
        }
    },
)

for hit in search_result["hits"]["hits"]:
    print(f"Score: {hit['_score']:.2f}")
    print(hit["_source"])
    print("-" * 80)

Score: 25.59
{'vehicle_class': 'Midsize Cars', 'electric_range_miles': 272.0, 'combined_mpge': 132.0, 'vehicle_name': '2022 Tesla Model 3 RWD'}
--------------------------------------------------------------------------------
Score: 25.59
{'vehicle_class': 'Midsize Cars', 'electric_range_miles': 272.0, 'combined_mpge': 132.0, 'vehicle_name': '2023 Tesla Model 3 RWD'}
--------------------------------------------------------------------------------
Score: 25.59
{'vehicle_class': 'Midsize Cars', 'electric_range_miles': 272.0, 'combined_mpge': 132.0, 'vehicle_name': '2024 Tesla Model 3 RWD'}
--------------------------------------------------------------------------------
Score: 25.59
{'vehicle_class': 'Midsize Cars', 'electric_range_miles': 309.0, 'combined_mpge': 114.0, 'vehicle_name': '2026 Tesla Model 3 Performance'}
--------------------------------------------------------------------------------
Score: 23.95
{'vehicle_class': 'Midsize Cars', 'electric_range_miles': 310.0, 'combined_mpge